# 📊 Data Exploration - License Plate Recognition

This notebook explores the dataset for license plate recognition.


In [ ]:
import sys
sys.path.append('../src')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import os

DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw' / 'images'
PROCESSED_DIR = DATA_DIR / 'processed'

print('Data directory:', DATA_DIR)
print('Images found:', len(list(RAW_DIR.glob('*.jpg'))) + len(list(RAW_DIR.glob('*.png'))))


In [ ]:
# Load and display sample images
image_files = list(RAW_DIR.glob('*.jpg'))[:12]

if image_files:
    fig, axes = plt.subplots(3, 4, figsize=(16, 9))
    for ax, img_path in zip(axes.flat, image_files):
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(img_path.name[:20], fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('../reports/figures/sample_images.png', dpi=150)
    plt.show()
    print(f'Displayed {len(image_files)} images')
else:
    print('No images found. Add images to data/raw/images/')


In [ ]:
# Image size distribution
image_files_all = list(RAW_DIR.glob('*.jpg')) + list(RAW_DIR.glob('*.png'))
sizes = []
for f in image_files_all:
    img = cv2.imread(str(f))
    if img is not None:
        h, w = img.shape[:2]
        sizes.append({'file': f.name, 'width': w, 'height': h, 'aspect': w/h})

if sizes:
    df = pd.DataFrame(sizes)
    print(df.describe())
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist2d(df['width'], df['height'], bins=20, cmap='Blues')
    ax[0].set_xlabel('Width'); ax[0].set_ylabel('Height'); ax[0].set_title('Image Size Distribution')
    ax[1].hist(df['aspect'], bins=20, color='steelblue', edgecolor='white')
    ax[1].set_xlabel('Aspect Ratio'); ax[1].set_title('Aspect Ratio Distribution')
    plt.tight_layout()
    plt.show()


In [ ]:
# Run detection on samples
from detection.detector import LicensePlateDetector
from preprocessing.plate_preprocessor import PlatePreprocessor

detector = LicensePlateDetector(model_path='../models/yolov8/best.pt')
preprocessor = PlatePreprocessor()

print('Detector info:', detector.get_model_info())

# Test on first image
if image_files_all:
    img = cv2.imread(str(image_files_all[0]))
    detections = detector.detect(img)
    print(f'Detections: {len(detections)}')
    for d in detections:
        print(f'  bbox={d["bbox"]}, conf={d["confidence"]:.2f}')
